In [5]:
import os
import pandas as pd

os.makedirs("data/silver", exist_ok=True)
os.makedirs("data/gold", exist_ok=True)

In [6]:
def read_file(path: str) -> pd.DataFrame:
  df = pd.read_csv(path)
  return df

In [25]:
silver_df = read_file("data/silver/customer_journey_silver.csv")

In [26]:
silver_df.head(5)

,SessionID,UserID,Timestamp,PageType,DeviceType,Country,ReferralSource,TimeOnPage_seconds,ItemsInCart,Purchased,funnel_order
0,session_0,user_2223,2025-01-20 22:53:34,home,Desktop,India,Social Media,55,0,0,1
1,session_1,user_2192,2025-02-26 12:57:10,home,Tablet,Germany,Email,99,0,0,1
2,session_1,user_2192,2025-02-26 12:59:11,product_page,Tablet,Germany,Email,121,0,0,2
3,session_10,user_2357,2025-05-17 22:11:37,home,Tablet,India,Direct,75,0,0,1
4,session_10,user_2357,2025-05-17 22:13:33,product_page,Tablet,India,Direct,116,0,0,2


In [49]:
months_mapping = {1: "jan", 2: "feb", 3: "mar", 4: "apr", 5: "may", 6: "jun", 7: "jul", 8: "aug", 9: "sep", 10: "oct", 11: "nov", 12: "dec"}
quarter_mapping = {1: 1, 2: 1, 3: 1, 4: 2, 5: 2, 6: 2, 7: 3, 8: 3, 9: 3, 10: 4, 11: 4, 12: 4}

def make_calendar_dimension(time_column: str, df: pd.DataFrame) -> pd.DataFrame:
  calendar_df = pd.DataFrame()
  calendar_df[time_column] = df[time_column]
  calendar_df[time_column] = pd.to_datetime(calendar_df[time_column])
  calendar_df["Year"] = calendar_df[time_column].dt.year
  calendar_df["Month"] = calendar_df[time_column].dt.month
  calendar_df["Day"] = calendar_df[time_column].dt.day
  calendar_df["Month_Name"] = calendar_df["Month"].map(months_mapping)
  calendar_df["Quarter"] = calendar_df["Month"].map(quarter_mapping)
  calendar_df.sort_values(by=time_column, inplace=True)
  calendar_df.drop(columns=[time_column], inplace=True)
  calendar_df.drop_duplicates(inplace=True)
  calendar_df.reset_index(drop=True, inplace=True)
  calendar_df["DateID"] = calendar_df.index + 1
  return calendar_df

In [52]:
def make_dimension(dimension: str, columns: list[str], df: pd.DataFrame) -> pd.DataFrame:
  dim = pd.DataFrame()
  dim[columns] = df[columns]
  df.drop_duplicates(inplace=True).reset_index(drop=True)
  df[f"{dimension}ID"] = df.index + 1
  return df

In [17]:
def merge_calendar(df: pd.DataFrame, calendar: pd.DataFrame) -> pd.DataFrame:
  df = df.copy()
  calendar = calendar.copy()
  df["data_ref"] = {
      "year": df["Timestamp"].dt.year,
      "month": df["Timestamp"].dt.month,
      "day": df["Timestamp"].dt.day
  }
  calendar["data_ref"] = {
      "year": calendar["Year"],
      "month": calendar["Month"],
      "day": calendar["Day"]
  }
  df = pd.merge(df, calendar, on="data_ref", how="left")
  df.drop(columns=["data_ref", "Timestamp", "Year", "Month", "Day"], inplace=True)
  return df

In [18]:
def merge_dimension(df: pd.DataFrame, dimension: pd.DataFrame, dimension_name: str) -> pd.DataFrame:
  df = df.copy()
  dimension = dimension.copy()
  merge_columns = dimension.columns.tolist()
  merge_columns.remove(f"{dimension_name}ID")
  df = pd.merge(df, dimension, on=merge_columns, how="left")
  df.drop(columns=[merge_columns], inplace=True)
  return df

In [14]:
def save_file(df: pd.DataFrame, path: str) -> None:
  df.to_csv(path, index=False)

In [50]:
calendar_dim = make_calendar_dimension("Timestamp", silver_df)
calendar_dim.head(5)

,Year,Month,Day,Month_Name,Quarter,DateID
0,2025,1,1,jan,1,1
1,2025,1,2,jan,1,2
2,2025,1,3,jan,1,3
3,2025,1,4,jan,1,4
4,2025,1,5,jan,1,5


In [51]:
calendar_dim.shape

(243, 6)